# LLM Providers & APIs

Large language models are built by a handful of organizations — **OpenAI**, **Google**, **Anthropic**, **xAI**, **DeepSeek**, **Meta**, **Mistral**, and others — but most developers access them through **provider APIs** rather than running billion-parameter models locally. A provider hosts the model on cloud GPUs, exposes an HTTP endpoint, and charges per token or per request.

This notebook is the **overview** for the provider landscape: what an LLM API is, how authentication works, and how providers compare at a high level. Each major provider has its own detailed notebook in this folder. Hands-on code lives in sibling folders under `03. LLM API Integration/`.

Understanding providers and their APIs is the foundation for everything that follows — prompt engineering, embeddings, RAG, and production applications all build on these HTTP calls.


---

## 1. What Is an LLM Provider API?

An **LLM provider** operates the infrastructure — model weights, GPUs, scaling, safety filters — and sells access through a **web API**. Your application sends an HTTPS request; the provider returns generated text (and metadata) in JSON.

```
  Your app                         Provider cloud
     |                                   |
     |  POST /v1/chat/completions        |
     |  Authorization: Bearer <API_KEY>  |
     |  { model, messages, temperature } |
     | --------------------------------> |
     |                                   |  Run inference
     |  { choices, usage, model }        |
     |  <--------------------------------|
```

You do **not** download model weights for API-based access. You send text in, receive text out, and pay for tokens consumed. The provider handles versioning, uptime, and scaling.

**Contrast with self-hosting:** Tools like **Ollama**, **vLLM**, and **llama.cpp** run models on your machine or server. They often expose an **OpenAI-compatible** local API, but you manage hardware, updates, and capacity yourself.


---

## 2. The Provider Landscape

The market clusters into a few categories:

| Category | Examples | How you access models |
|----------|----------|----------------------|
| **Frontier API providers** | OpenAI, Google (Gemini), Anthropic, xAI, DeepSeek | Direct REST API + official SDK |
| **Open-weight hosts** | Meta (Llama), Mistral, Hugging Face | Download weights or use hosted inference APIs |
| **Cloud marketplaces** | Azure OpenAI, AWS Bedrock, Google Vertex AI | Provider models through your cloud account |
| **Local / edge** | Ollama, LM Studio, llama.cpp | Run on laptop or private server |

### Notebooks in this folder

| Notebook | Provider | Focus |
|----------|----------|-------|
| `02. OpenAI.ipynb` | OpenAI | GPT models, chat completions API, ecosystem |
| `03. Google Gemini.ipynb` | Google | Gemini models, multimodal API, AI Studio vs Vertex |
| `04. Anthropic Claude.ipynb` | Anthropic | Claude family, Messages API, long context |
| `05. X's Grok.ipynb` | xAI | Grok models, real-time data, xAI API |
| `06. DeepSeek.ipynb` | DeepSeek | Cost-efficient models, reasoning (R1), open weights |
| `07. Other LLM Providers.ipynb` | Various | Mistral, Bedrock, Azure, Ollama, multi-provider patterns |

We integrate **OpenAI**, **Gemini**, and **Claude** in code in this course. Grok and DeepSeek are covered in theory here; you can add API keys and experiment using the same patterns.


---

## 3. Common API Concepts Across Providers

These ideas apply regardless of which provider you use.

### 3.1 Messages and conversation state

APIs are **stateless** — the model has no memory between calls. You must resend the full conversation history on every request for multi-turn chat. Long histories consume more input tokens and cost more.

### 3.2 Inference parameters

All providers accept parameters that control generation (see GenAI Fundamentals — *Inference Parameters and Text Generation*):

- **temperature** — randomness (0 = focused, higher = more creative)
- **max tokens** — limit on output length
- **stop sequences** — strings that halt generation

### 3.3 Streaming

Streaming returns partial output as tokens are generated. The final text and total cost are the same as non-streaming; perceived latency is lower for interactive UIs.

### 3.4 Token billing

You pay for **input tokens** (prompt + history) and **output tokens** (completion). Responses include usage counts. **Rate limits** apply per minute at the account or tier level.

### 3.5 Errors

| HTTP code | Meaning | Typical action |
|-----------|---------|----------------|
| 401 | Invalid API key | Check `.env` |
| 429 | Rate limit exceeded | Retry with exponential backoff |
| 400 | Bad request (wrong model, malformed body) | Fix parameters |
| 500 | Provider server error | Retry; check status page |


---

## 4. Comparing Provider APIs Side by Side

| Feature | OpenAI | Google Gemini | Anthropic Claude |
|---------|--------|---------------|------------------|
| **Python package** | `openai` | `google-genai` | `anthropic` |
| **Client init** | `OpenAI()` | `genai.Client(api_key=...)` | `Anthropic()` |
| **Main call** | `chat.completions.create()` | `models.generate_content()` | `messages.create()` |
| **System prompt** | `role: system` in messages | `system_instruction` or turns | `system=` parameter |
| **User/assistant** | `messages` list | `contents` list | `messages` list |
| **Streaming** | `stream=True` | `generate_content_stream()` | `stream=True` |
| **Max output** | `max_tokens` | `max_output_tokens` | `max_tokens` |
| **Usage fields** | `prompt_tokens`, `completion_tokens` | `usage_metadata` | `input_tokens`, `output_tokens` |
| **Auth** | `Bearer` token | API key in client | `x-api-key` header |

Despite different SDKs, the **mental model is identical**:

1. Create an authenticated client
2. Choose a model ID
3. Send instructions + user input
4. Read generated text and token usage from the response

See each provider notebook for provider-specific strengths and API details.


---

## 5. API Keys and Authentication

Every provider uses **secret API keys** to authenticate requests. Keys are tied to your billing account and must be protected.

| Provider | Environment variable | Where to get a key |
|----------|---------------------|-------------------|
| OpenAI | `OPENAI_API_KEY` | [platform.openai.com/api-keys](https://platform.openai.com/api-keys) |
| Google Gemini | `GOOGLE_API_KEY` | [aistudio.google.com/apikey](https://aistudio.google.com/apikey) |
| Anthropic | `ANTHROPIC_API_KEY` | [console.anthropic.com](https://console.anthropic.com) |

**Setup in this project:**

1. Copy `example.env` to `.env` at the **repo root**
2. Paste your keys (never commit `.env`)
3. Scripts use `llm_env.py`, which walks up the directory tree to find `.env` automatically

**Security rules:**

- Never hard-code keys in notebooks or `.py` files
- Add `.env` to `.gitignore`
- Use separate keys for development and production
- Set spending limits in each provider's dashboard
- Rotate keys immediately if exposed


---

## 6. Choosing a Provider

No single provider wins every task. Use this as a starting guide:

| Priority | Consider |
|----------|----------|
| **Ecosystem and tutorials** | OpenAI — largest community, most examples |
| **Multimodal (image/audio/video)** | Gemini — native multimodal API |
| **Long documents** | Claude or Gemini — large context windows |
| **Coding assistance** | All three are strong; compare on your codebase |
| **Cost at scale** | DeepSeek, smaller models, or open-weight hosting |
| **Real-time / social context** | Grok (xAI) — X platform integration |
| **Enterprise compliance** | Azure OpenAI, Vertex AI, Bedrock — cloud IAM, private networking |
| **Privacy / offline** | Ollama or self-hosted vLLM |

Many production systems are **multi-provider**: a router sends simple queries to a cheap model and complex queries to a frontier model, or falls back if one API is down.


---

## 7. Hands-On Code in This Course

| Project / script | Provider | Demonstrates |
|------------------|----------|--------------|
| `02. OpenAI/01. Console Application/` | OpenAI | Direct SDK: chat, streaming, tiktoken |
| `02. OpenAI/02. Fast API Application/` | OpenAI | FastAPI: HTTP API, retries, SSE |
| `03. Google Gemini/01. Console Application/` | Gemini | Direct SDK: chat, streaming, token count |
| `03. Google Gemini/02. Fast API Application/` | Gemini | FastAPI: HTTP API, retries, SSE |
| `04. Anthropic Claude/01. Console Application/` | Claude | Direct SDK: messages, streaming, token count |
| `04. Anthropic Claude/02. Fast API Application/` | Claude | FastAPI: HTTP API, retries, SSE |

```bash
# OpenAI — console
python "04. Generative AI/03. LLM API Integration/02. OpenAI/01. Console Application/01_chat_completion.py"

# Gemini — FastAPI (port 8011)
cd "04. Generative AI/03. LLM API Integration/03. Google Gemini/02. Fast API Application"
uvicorn app.main:app --reload --port 8011

# Claude — FastAPI (port 8012)
cd "04. Generative AI/03. LLM API Integration/04. Anthropic Claude/02. Fast API Application"
uvicorn app.main:app --reload --port 8012
```


---

## 8. Summary

**LLM providers** host frontier models and sell access through HTTP APIs. You authenticate with API keys, send structured prompts, and receive generated text billed by the token.

Each provider has its own SDK and naming conventions, but the workflow is the same: create a client, pick a model, send messages, read the response. Use the **individual provider notebooks** in this folder for depth on OpenAI, Gemini, Claude, Grok, DeepSeek, and others.

Next steps in the curriculum cover prompt engineering, embeddings, and RAG — all built on these provider API calls.
